# Round 3 Options Alpha Scan (VEV + Underlying)

This notebook is now focused only on VEV options and the underlying `VELVETFRUIT_EXTRACT`.

Focus:
- implied volatility vs realized volatility (IV-RV)
- smile residual (cross-strike mispricing)
- Black-Scholes Greeks (delta, gamma, theta)
- delta-hedged residual and simple gamma-scalping proxy

Scoring uses spread-adjusted net PnL proxies so ranking is closer to executable behavior.

In [15]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

ROUND = 3
ROOT = Path('.')
PRICE_FILES = sorted(ROOT.glob(f'prices_round_{ROUND}_day_*.csv'))
TRADE_FILES = sorted(ROOT.glob(f'trades_round_{ROUND}_day_*.csv'))

if not PRICE_FILES:
    raise FileNotFoundError('No round-3 price files found in current folder.')

day_re = re.compile(r'day_(-?\d+)\.csv$')

def _day_from_name(path: Path) -> int:
    m = day_re.search(path.name)
    if m is None:
        raise ValueError(f'Cannot parse day from filename: {path.name}')
    return int(m.group(1))

price_frames = []
for fp in PRICE_FILES:
    day = _day_from_name(fp)
    df = pd.read_csv(fp, sep=';')
    df['day'] = day

    for col in [
        'timestamp', 'mid_price',
        'bid_price_1', 'bid_volume_1',
        'ask_price_1', 'ask_volume_1',
        'bid_price_2', 'bid_volume_2',
        'ask_price_2', 'ask_volume_2',
        'bid_price_3', 'bid_volume_3',
        'ask_price_3', 'ask_volume_3',
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    df['product'] = df['product'].astype(str).str.strip().str.upper()
    price_frames.append(df)

prices = pd.concat(price_frames, ignore_index=True)
prices = prices.dropna(subset=['timestamp', 'product']).copy()
prices['timestamp'] = prices['timestamp'].astype(float)

trade_frames = []
for fp in TRADE_FILES:
    day = _day_from_name(fp)
    td = pd.read_csv(fp, sep=';')
    td['day'] = day

    for col in ['timestamp', 'price', 'quantity']:
        if col in td.columns:
            td[col] = pd.to_numeric(td[col], errors='coerce')

    if 'symbol' in td.columns:
        td['symbol'] = td['symbol'].astype(str).str.strip().str.upper()
    trade_frames.append(td)

trades = pd.concat(trade_frames, ignore_index=True) if trade_frames else pd.DataFrame()

print(f'Loaded {len(prices):,} price rows across {prices["product"].nunique()} products')
if not trades.empty:
    print(f'Loaded {len(trades):,} trade rows across {trades["symbol"].nunique()} symbols')
else:
    print('No trade rows loaded (trade-based features will be skipped).')

display(prices.head(3))

Loaded 360,000 price rows across 12 products
Loaded 4,048 trade rows across 12 symbols


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
0,0,0.0,VEV_5400,22,25,NaN,NaN,NaN,NaN,24,25,NaN,NaN,NaN,NaN,23.0,0.0
1,0,0.0,VEV_6500,0,16,NaN,NaN,NaN,NaN,1,16,NaN,NaN,NaN,NaN,0.5,0.0
2,0,0.0,VEV_5500,8,25,NaN,NaN,NaN,NaN,9,25,NaN,NaN,NaN,NaN,8.5,0.0


In [16]:
# Build options-only feature table: VEV options + VELVETFRUIT_EXTRACT
from math import erf, exp, log, pi, sqrt

UNDERLYING = 'VELVETFRUIT_EXTRACT'
TTM_YEARS = 7.0 / 365.0

def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def norm_pdf(x: float) -> float:
    return exp(-0.5 * x * x) / sqrt(2.0 * pi)

def bs_call_price(spot: float, strike: float, ttm: float, sigma: float, rate: float = 0.0) -> float:
    if sigma <= 0.0 or ttm <= 0.0:
        return max(spot - strike, 0.0)
    vol_t = sigma * sqrt(ttm)
    d1 = (log(spot / strike) + (rate + 0.5 * sigma * sigma) * ttm) / vol_t
    d2 = d1 - vol_t
    return spot * norm_cdf(d1) - strike * exp(-rate * ttm) * norm_cdf(d2)

def implied_vol_call(price: float, spot: float, strike: float, ttm: float, rate: float = 0.0) -> float:
    intrinsic = max(spot - strike * exp(-rate * ttm), 0.0)
    if not (intrinsic < price < spot):
        return np.nan

    lo, hi = 1e-4, 5.0
    f_lo = bs_call_price(spot, strike, ttm, lo, rate) - price
    f_hi = bs_call_price(spot, strike, ttm, hi, rate) - price

    for _ in range(12):
        if f_lo * f_hi <= 0:
            break
        hi *= 1.5
        f_hi = bs_call_price(spot, strike, ttm, hi, rate) - price
    if f_lo * f_hi > 0:
        return np.nan

    for _ in range(80):
        mid = 0.5 * (lo + hi)
        f_mid = bs_call_price(spot, strike, ttm, mid, rate) - price
        if abs(f_mid) < 1e-7:
            return mid
        if f_lo * f_mid <= 0:
            hi = mid
            f_hi = f_mid
        else:
            lo = mid
            f_lo = f_mid
    return 0.5 * (lo + hi)

def bs_delta_gamma_theta(spot: float, strike: float, ttm: float, sigma: float, rate: float = 0.0):
    if sigma <= 0.0 or ttm <= 0.0 or spot <= 0.0 or strike <= 0.0:
        return np.nan, np.nan, np.nan
    vol_t = sigma * sqrt(ttm)
    d1 = (log(spot / strike) + (rate + 0.5 * sigma * sigma) * ttm) / vol_t
    d2 = d1 - vol_t

    delta = norm_cdf(d1)
    gamma = norm_pdf(d1) / (spot * vol_t)
    theta = -(spot * norm_pdf(d1) * sigma) / (2.0 * sqrt(ttm)) - rate * strike * exp(-rate * ttm) * norm_cdf(d2)
    return delta, gamma, theta

# Underlying snapshot
spot_df = prices[prices['product'] == UNDERLYING][['day', 'timestamp', 'mid_price']].copy()
spot_df = spot_df.rename(columns={'mid_price': 'spot'})
spot_df = spot_df.dropna(subset=['spot']).sort_values(['day', 'timestamp'])

spot_df['spot_ret_1'] = spot_df.groupby('day')['spot'].shift(-1) - spot_df['spot']
spot_df['log_ret'] = spot_df.groupby('day')['spot'].transform(lambda s: np.log(s).diff())

ticks_per_day = float(spot_df.groupby('day')['timestamp'].nunique().median())
if not np.isfinite(ticks_per_day) or ticks_per_day <= 0:
    ticks_per_day = 1000.0
dt_year = 1.0 / (252.0 * ticks_per_day)

spot_df['rv_30'] = (
    spot_df.groupby('day')['log_ret']
    .rolling(30, min_periods=12)
    .std(ddof=0)
    .reset_index(level=0, drop=True)
    / sqrt(dt_year)
)

# Options table
opt = prices[prices['product'].str.match(r'^VEV_\d+$', na=False)].copy()
opt['strike'] = pd.to_numeric(opt['product'].str.extract(r'(\d+)')[0], errors='coerce')
opt = opt.dropna(subset=['strike', 'mid_price']).copy()
opt = opt.rename(columns={'mid_price': 'option_mid'})
opt = opt.sort_values(['day', 'product', 'timestamp'])

# Join spot and realized vol
df_opt = opt.merge(
    spot_df[['day', 'timestamp', 'spot', 'spot_ret_1', 'rv_30']],
    on=['day', 'timestamp'],
    how='inner'
)

df_opt['spread'] = df_opt['ask_price_1'] - df_opt['bid_price_1']
df_opt['moneyness'] = df_opt['strike'] / df_opt['spot'] - 1.0

df_opt['ret_1'] = df_opt.groupby(['day', 'product'])['option_mid'].shift(-1) - df_opt['option_mid']
df_opt['ret_2'] = df_opt.groupby(['day', 'product'])['option_mid'].shift(-2) - df_opt['option_mid']

# IV and Greeks
df_opt['iv'] = df_opt.apply(
    lambda r: implied_vol_call(
        float(r['option_mid']),
        float(r['spot']),
        float(r['strike']),
        TTM_YEARS,
        0.0,
    ),
    axis=1,
)

greeks = df_opt.apply(
    lambda r: bs_delta_gamma_theta(
        float(r['spot']),
        float(r['strike']),
        TTM_YEARS,
        float(r['iv']) if pd.notna(r['iv']) else np.nan,
        0.0,
    ),
    axis=1,
)
df_opt[['delta', 'gamma', 'theta']] = pd.DataFrame(greeks.tolist(), index=df_opt.index)

# IV-RV spread
df_opt['iv_rv_spread'] = df_opt['iv'] - df_opt['rv_30']

# Smile residual by (day,timestamp): iv - fitted_quadratic(moneyness)
def add_smile_residual(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy()
    valid = g.dropna(subset=['moneyness', 'iv'])
    if len(valid) < 4:
        g['smile_resid'] = np.nan
        return g
    x = valid['moneyness'].to_numpy()
    y = valid['iv'].to_numpy()
    a2, a1, a0 = np.polyfit(x, y, 2)
    g['smile_fit_iv'] = a2 * g['moneyness'] ** 2 + a1 * g['moneyness'] + a0
    g['smile_resid'] = g['iv'] - g['smile_fit_iv']
    return g

df_opt = df_opt.groupby(['day', 'timestamp'], group_keys=False).apply(add_smile_residual)

# Delta-hedged residual return and gamma-scalp proxy
df_opt['delta_hedged_resid'] = df_opt['ret_1'] - df_opt['delta'] * df_opt['spot_ret_1']
df_opt['realized_move'] = df_opt['spot_ret_1'].abs()
df_opt['implied_move'] = df_opt['iv'] * df_opt['spot'] * sqrt(dt_year)
df_opt['gamma_scalp_proxy'] = df_opt['gamma'] * (df_opt['realized_move'] - df_opt['implied_move'])

# Keep usable rows
df_opt = df_opt.dropna(subset=['ret_1', 'spot', 'option_mid']).copy()

display(
    df_opt[[
        'day', 'timestamp', 'product', 'strike', 'spot', 'option_mid',
        'iv', 'rv_30', 'iv_rv_spread', 'delta', 'gamma', 'theta',
        'smile_resid', 'delta_hedged_resid', 'gamma_scalp_proxy', 'ret_1',
    ]].head(12)
)
print(f'Rows in options feature table: {len(df_opt):,}')
print(f"Unique options: {df_opt['product'].nunique()}")

C:\Users\tolan\AppData\Local\Temp\ipykernel_28420\406648308.py:145: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_opt = df_opt.groupby(['day', 'timestamp'], group_keys=False).apply(add_smile_residual)


,day,timestamp,product,strike,spot,option_mid,iv,rv_30,iv_rv_spread,delta,gamma,theta,smile_resid,delta_hedged_resid,gamma_scalp_proxy,ret_1
0,0,0.0,VEV_4000,4000,5250.0,1250.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.5
1,0,100.0,VEV_4000,4000,5250.5,1250.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,0,200.0,VEV_4000,4000,5250.5,1250.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,0,300.0,VEV_4000,4000,5250.5,1250.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,0,400.0,VEV_4000,4000,5250.5,1250.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0
5,0,500.0,VEV_4000,4000,5249.5,1249.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0
6,0,600.0,VEV_4000,4000,5248.5,1248.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
7,0,700.0,VEV_4000,4000,5248.5,1248.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0
8,0,800.0,VEV_4000,4000,5247.5,1247.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0
9,0,900.0,VEV_4000,4000,5246.5,1246.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


Rows in options feature table: 299,970
Unique options: 10


In [17]:
# Candidate alpha library (options + underlying-vol aware)
df_alpha = df_opt.copy()

alpha_map = {
    # 1) IV-RV relationship
    'iv_rv_mr': -df_alpha['iv_rv_spread'],
    'iv_rv_mom': df_alpha['iv_rv_spread'],

    # 2) Smile mispricing across strikes
    'smile_resid_mr': -df_alpha['smile_resid'],
    'smile_resid_mom': df_alpha['smile_resid'],

    # 3) Delta-hedged residual
    'delta_hedge_resid_mr': -df_alpha['delta_hedged_resid'],
    'delta_hedge_resid_mom': df_alpha['delta_hedged_resid'],

    # 4) Theta carry (very rough proxy)
    'theta_carry_short': -df_alpha['theta'],

    # 5) Gamma scalping proxy
    'gamma_scalp': df_alpha['gamma_scalp_proxy'],
}

for name, s in alpha_map.items():
    df_alpha[name] = s

alpha_cols = list(alpha_map.keys())
print(f'Built {len(alpha_cols)} options-focused alpha candidates')
alpha_cols

Built 8 options-focused alpha candidates


['iv_rv_mr',
 'iv_rv_mom',
 'smile_resid_mr',
 'smile_resid_mom',
 'delta_hedge_resid_mr',
 'delta_hedge_resid_mom',
 'theta_carry_short',
 'gamma_scalp']

In [18]:
# Evaluation helpers (options-only, spread-adjusted)
def zscore_by_group(series: pd.Series, day: pd.Series, product: pd.Series) -> pd.Series:
    g = series.groupby([day, product])
    mu = g.transform('mean')
    sd = g.transform('std').replace(0, np.nan)
    return (series - mu) / sd

def make_hysteresis_position(sig_z: pd.Series, entry: float, exit_: float, cap: float) -> pd.Series:
    pos = 0.0
    out = []
    for s in sig_z.to_numpy():
        if np.isnan(s):
            out.append(pos)
            continue
        if abs(s) >= entry:
            pos = float(np.clip(s, -cap, cap))
        elif abs(s) <= exit_:
            pos = 0.0
        out.append(pos)
    return pd.Series(out, index=sig_z.index)

def evaluate_candidate(
    base: pd.DataFrame,
    signal_col: str,
    horizon_col: str = 'ret_1',
    clip_z: float = 2.0,
    entry: float = 0.8,
    exit_: float = 0.4,
    pos_cap: float = 2.0,
    spread_cost_weight: float = 0.5,
) -> dict:
    cols = ['day', 'timestamp', 'product', 'spread', horizon_col, signal_col]
    t = base[cols].copy()
    t = t.dropna(subset=[signal_col, horizon_col]).copy()
    if t.empty:
        return {
            'alpha': signal_col,
            'rows': 0,
            'gross_pnl': np.nan,
            'net_pnl': np.nan,
            'gross_sharpe': np.nan,
            'net_sharpe': np.nan,
            'turnover': np.nan,
            'active_rate': np.nan,
        }

    t['sig_z'] = zscore_by_group(t[signal_col], t['day'], t['product']).clip(-clip_z, clip_z)
    t = t.sort_values(['day', 'product', 'timestamp']).copy()
    t['position'] = (
        t.groupby(['day', 'product'], group_keys=False)['sig_z']
        .apply(lambda s: make_hysteresis_position(s, entry, exit_, pos_cap))
    )
    t['prev_position'] = t.groupby(['day', 'product'])['position'].shift(1).fillna(0.0)
    t['turnover'] = (t['position'] - t['prev_position']).abs()

    t['gross_tick'] = t['position'] * t[horizon_col]
    t['half_spread'] = (t['spread'] / 2.0).abs().fillna(0.0)
    t['exec_cost'] = spread_cost_weight * t['turnover'] * t['half_spread']
    t['net_tick'] = t['gross_tick'] - t['exec_cost']

    gstd = t['gross_tick'].std(ddof=0)
    nstd = t['net_tick'].std(ddof=0)

    return {
        'alpha': signal_col,
        'rows': int(len(t)),
        'gross_pnl': float(t['gross_tick'].sum()),
        'net_pnl': float(t['net_tick'].sum()),
        'gross_sharpe': float(t['gross_tick'].mean() / gstd) if gstd > 0 else np.nan,
        'net_sharpe': float(t['net_tick'].mean() / nstd) if nstd > 0 else np.nan,
        'turnover': float(t['turnover'].mean()),
        'active_rate': float((t['position'].abs() > 0).mean()),
    }

In [19]:
# Run options alpha sweep (ret_1 and ret_2 horizons)
rows = []
for c in alpha_cols:
    r1 = evaluate_candidate(df_alpha, c, horizon_col='ret_1', spread_cost_weight=0.5)
    r1['horizon'] = 'ret_1'
    rows.append(r1)

    r2 = evaluate_candidate(df_alpha, c, horizon_col='ret_2', spread_cost_weight=0.5)
    r2['horizon'] = 'ret_2'
    rows.append(r2)

summary = pd.DataFrame(rows).sort_values(['net_pnl', 'net_sharpe'], ascending=False).reset_index(drop=True)

print('Top options alpha candidates (VEV only)')
display(summary.head(20))

print('Best per family (across horizons)')
best_by_family = (
    summary.sort_values(['alpha', 'net_pnl', 'net_sharpe'], ascending=[True, False, False])
    .groupby('alpha', as_index=False)
    .head(1)
    .sort_values('net_pnl', ascending=False)
 )
display(best_by_family)

# Product-level ranking for the best few families
top_families = best_by_family['alpha'].head(3).tolist()
if top_families:
    by_product_rows = []
    for fam in top_families:
        t = df_alpha[['day', 'timestamp', 'product', 'spread', 'ret_1', fam]].dropna().copy()
        if t.empty:
            continue
        t['sig_z'] = zscore_by_group(t[fam], t['day'], t['product']).clip(-2.0, 2.0).fillna(0.0)
        t = t.sort_values(['day', 'product', 'timestamp'])
        t['position'] = t.groupby(['day', 'product'])['sig_z'].shift(0).fillna(0.0)
        t['prev_position'] = t.groupby(['day', 'product'])['position'].shift(1).fillna(0.0)
        t['turnover'] = (t['position'] - t['prev_position']).abs()
        t['net_tick'] = t['position'] * t['ret_1'] - 0.5 * t['turnover'] * (t['spread'] / 2.0).abs().fillna(0.0)

        agg = t.groupby('product').agg(
            rows=('net_tick', 'size'),
            net_pnl=('net_tick', 'sum'),
            mean_tick=('net_tick', 'mean'),
            turnover=('turnover', 'mean'),
        ).reset_index()
        agg['alpha'] = fam
        by_product_rows.append(agg)

    if by_product_rows:
        by_product = pd.concat(by_product_rows, ignore_index=True)
        print('Top-family performance by option symbol')
        display(by_product.sort_values(['alpha', 'net_pnl'], ascending=[True, False]))

Top options alpha candidates (VEV only)


,alpha,rows,gross_pnl,net_pnl,gross_sharpe,net_sharpe,turnover,active_rate,horizon
0,iv_rv_mr,251433,5273.589503,-31227.154035,0.024678,-0.115425,0.113340,0.555377,ret_2
1,iv_rv_mr,251458,4939.989074,-31567.647383,0.028334,-0.131594,0.113339,0.555393,ret_1
2,iv_rv_mom,251458,-4939.989074,-41447.625530,-0.028334,-0.152113,0.113339,0.555393,ret_1
3,iv_rv_mom,251433,-5273.589503,-41774.333042,-0.024678,-0.139935,0.113340,0.555377,ret_2
4,theta_carry_short,251736,-6414.795631,-42110.173657,-0.029344,-0.138735,0.095137,0.561112,ret_2
5,theta_carry_short,251761,-7236.441149,-42931.174852,-0.039970,-0.154119,0.095121,0.561104,ret_1
6,smile_resid_mr,251736,5066.442699,-96009.258002,0.022351,-0.274504,0.497889,0.729335,ret_2
7,smile_resid_mr,251761,4884.521644,-96228.305955,0.027671,-0.301137,0.498002,0.729335,ret_1
8,smile_resid_mom,251761,-4884.521644,-105997.349243,-0.027671,-0.314643,0.498002,0.729335,ret_1
9,smile_resid_mom,251736,-5066.442699,-106142.143400,-0.022351,-0.290321,0.497889,0.729335,ret_2


Best per family (across horizons)


,alpha,rows,gross_pnl,net_pnl,gross_sharpe,net_sharpe,turnover,active_rate,horizon
0,iv_rv_mr,251433,5273.589503,-31227.154035,0.024678,-0.115425,0.113340,0.555377,ret_2
2,iv_rv_mom,251458,-4939.989074,-41447.625530,-0.028334,-0.152113,0.113339,0.555393,ret_1
4,theta_carry_short,251736,-6414.795631,-42110.173657,-0.029344,-0.138735,0.095137,0.561112,ret_2
6,smile_resid_mr,251736,5066.442699,-96009.258002,0.022351,-0.274504,0.497889,0.729335,ret_2
8,smile_resid_mom,251761,-4884.521644,-105997.349243,-0.027671,-0.314643,0.498002,0.729335,ret_1
10,delta_hedge_resid_mom,251761,34137.608151,-110571.058457,0.173115,-0.363882,0.866140,0.489528,ret_1
12,gamma_scalp,251736,6193.204455,-133354.485232,0.026530,-0.397752,0.771469,0.577299,ret_2
14,delta_hedge_resid_mr,251736,-18081.718582,-162779.553421,-0.083670,-0.416596,0.866153,0.489509,ret_2


Top-family performance by option symbol


,product,rows,net_pnl,mean_tick,turnover,alpha
18,VEV_6000,29961,-1025.710592,-0.034235,0.136939,iv_rv_mom
19,VEV_6500,29961,-1026.095771,-0.034248,0.136991,iv_rv_mom
17,VEV_5500,29961,-1283.915307,-0.042853,0.140197,iv_rv_mom
16,VEV_5400,29961,-1524.719133,-0.050890,0.139004,iv_rv_mom
15,VEV_5300,29961,-2365.524311,-0.078953,0.140278,iv_rv_mom
14,VEV_5200,29961,-3255.993145,-0.108674,0.141252,iv_rv_mom
13,VEV_5100,29961,-5009.931127,-0.167215,0.148007,iv_rv_mom
12,VEV_5000,29937,-8754.883847,-0.292444,0.180981,iv_rv_mom
10,VEV_4000,2983,-14686.280148,-4.923326,0.871659,iv_rv_mom
11,VEV_4500,8811,-17840.063083,-2.024749,0.485972,iv_rv_mom


## Next extensions

- Add product-specific ranking so options and underlying are evaluated separately.
- Add hysteresis (entry/exit) instead of raw z-positioning for lower turnover.
- Add pair/relative-value alphas: e.g., 5400 versus local strike basket, but with execution-aware fill simulation.
- Port top 1-2 alphas into a new trading file and compare to official backtest output.

## Execution-Aware Extensions

The next two cells move from proxy scoring toward tradeable behavior:

1. Top-of-book execution simulation for top alpha families.
2. Delta-neutral cross-strike smile portfolio with turnover/spread costs.

In [22]:
# Top-of-book execution simulator for options alpha families
def simulate_tob_execution(
    base: pd.DataFrame,
    signal_col: str,
    horizon_col: str = 'ret_1',
    entry: float = 0.8,
    exit_: float = 0.4,
    clip_z: float = 2.0,
    pos_cap: float = 2.0,
    target_unit: int = 20,
    max_slice: int = 12,
) -> tuple[dict, pd.DataFrame]:
    req = [
        'day', 'timestamp', 'product', signal_col,
        'option_mid', 'bid_price_1', 'ask_price_1', 'bid_volume_1', 'ask_volume_1',
    ]
    t = base[req].dropna(subset=['option_mid']).copy()
    t = t.dropna(subset=[signal_col]).sort_values(['day', 'product', 'timestamp']).copy()

    if t.empty:
        return {'alpha': signal_col, 'rows': 0}, pd.DataFrame()

    t['sig_z'] = zscore_by_group(t[signal_col], t['day'], t['product']).clip(-clip_z, clip_z)
    t['signal_pos'] = (
        t.groupby(['day', 'product'], group_keys=False)['sig_z']
        .apply(lambda s: make_hysteresis_position(s, entry, exit_, pos_cap))
    )
    t['target_pos'] = np.round(t['signal_pos'] * target_unit).astype(float)

    records = []
    for (day, product), g in t.groupby(['day', 'product'], sort=False):
        g = g.sort_values('timestamp').copy()
        inv = 0.0
        cash = 0.0
        prev_equity = 0.0

        for _, r in g.iterrows():
            target = float(r['target_pos'])
            need = target - inv

            bid = float(r['bid_price_1']) if pd.notna(r['bid_price_1']) else np.nan
            ask = float(r['ask_price_1']) if pd.notna(r['ask_price_1']) else np.nan
            bvol = float(r['bid_volume_1']) if pd.notna(r['bid_volume_1']) else 0.0
            avol = float(r['ask_volume_1']) if pd.notna(r['ask_volume_1']) else 0.0

            fill = 0.0
            fill_px = np.nan

            if need > 0 and pd.notna(ask):
                fill = min(need, max_slice, avol)
                fill_px = ask
            elif need < 0 and pd.notna(bid):
                fill = -min(-need, max_slice, bvol)
                fill_px = bid

            if fill != 0.0 and pd.notna(fill_px):
                cash -= fill * fill_px
                inv += fill

            mid = float(r['option_mid'])
            equity = cash + inv * mid
            pnl_tick = equity - prev_equity
            prev_equity = equity

            records.append({
                'day': day,
                'timestamp': float(r['timestamp']),
                'product': product,
                'alpha': signal_col,
                'signal_pos': float(r['signal_pos']),
                'target_pos': target,
                'inventory': inv,
                'fill_qty': fill,
                'fill_px': fill_px,
                'mid': mid,
                'equity': equity,
                'pnl_tick': pnl_tick,
            })

    sim = pd.DataFrame(records)
    if sim.empty:
        return {'alpha': signal_col, 'rows': 0}, sim

    std = sim['pnl_tick'].std(ddof=0)
    metrics = {
        'alpha': signal_col,
        'rows': int(len(sim)),
        'net_pnl_exec': float(sim['pnl_tick'].sum()),
        'sharpe_exec': float(sim['pnl_tick'].mean() / std) if std > 0 else np.nan,
        'avg_abs_inventory': float(sim['inventory'].abs().mean()),
        'avg_fill_size': float(sim['fill_qty'].abs().mean()),
        'active_fill_rate': float((sim['fill_qty'] != 0).mean()),
    }
    return metrics, sim

if 'best_by_family' not in globals() or best_by_family.empty:
    raise ValueError('Run the sweep cell first to create best_by_family.')

top_exec_candidates = best_by_family['alpha'].head(5).tolist()
exec_rows = []
exec_sims = {}
for alpha_name in top_exec_candidates:
    m, s = simulate_tob_execution(df_alpha, alpha_name, horizon_col='ret_1')
    exec_rows.append(m)
    exec_sims[alpha_name] = s

exec_summary = pd.DataFrame(exec_rows).sort_values('net_pnl_exec', ascending=False).reset_index(drop=True)
print('Execution-aware ranking (top-of-book fill model)')
display(exec_summary)

if not exec_summary.empty:
    lead = exec_summary.iloc[0]['alpha']
    print(f'Lead alpha under execution model: {lead}')
    display(exec_sims[lead].head(20))

Execution-aware ranking (top-of-book fill model)


,alpha,rows,net_pnl_exec,sharpe_exec,avg_abs_inventory,avg_fill_size,active_fill_rate
0,theta_carry_short,251761,-885832.5,-0.203357,13.657346,1.397131,0.294299
1,iv_rv_mr,251458,-1014927.5,-0.220975,13.563020,1.992058,0.390761
2,iv_rv_mom,251458,-1106853.5,-0.230312,13.577647,2.013581,0.390415
3,smile_resid_mr,251761,-1981291.0,-0.415423,15.688562,5.321269,0.623266
4,smile_resid_mom,251761,-2090892.0,-0.431604,15.726888,5.377231,0.622837


Lead alpha under execution model: theta_carry_short


,day,timestamp,product,alpha,signal_pos,target_pos,inventory,fill_qty,fill_px,mid,equity,pnl_tick
0,0,1600.0,VEV_4000,theta_carry_short,0.0,0.0,0.0,0.0,NaN,1246.5,0.0,0.0
1,0,3300.0,VEV_4000,theta_carry_short,0.0,0.0,0.0,0.0,NaN,1245.5,0.0,0.0
2,0,3800.0,VEV_4000,theta_carry_short,0.0,0.0,0.0,0.0,NaN,1244.5,0.0,0.0
3,0,3900.0,VEV_4000,theta_carry_short,0.0,0.0,0.0,0.0,NaN,1243.5,0.0,0.0
4,0,8800.0,VEV_4000,theta_carry_short,0.0,0.0,0.0,0.0,NaN,1231.0,0.0,0.0
5,0,9100.0,VEV_4000,theta_carry_short,0.0,0.0,0.0,0.0,NaN,1231.0,0.0,0.0
6,0,9300.0,VEV_4000,theta_carry_short,0.0,0.0,0.0,0.0,NaN,1231.5,0.0,0.0
7,0,9900.0,VEV_4000,theta_carry_short,0.0,0.0,0.0,0.0,NaN,1234.0,0.0,0.0
8,0,10400.0,VEV_4000,theta_carry_short,2.0,40.0,8.0,8.0,1244.0,1238.5,-44.0,-44.0
9,0,13500.0,VEV_4000,theta_carry_short,0.0,0.0,0.0,-8.0,1232.0,1242.5,-96.0,-52.0


In [21]:
# Delta-neutral cross-strike smile residual portfolio
def delta_neutral_smile_portfolio(
    base: pd.DataFrame,
    signal_col: str = 'smile_resid',
    contracts_scale: float = 80.0,
    turnover_cost_weight: float = 0.5,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    req = [
        'day', 'timestamp', 'product', 'delta', 'spread', 'ret_1', signal_col,
    ]
    t = base[req].dropna(subset=['delta', signal_col, 'ret_1']).copy()
    t = t.sort_values(['day', 'timestamp', 'product'])

    if t.empty:
        return pd.DataFrame(), pd.DataFrame()

    rows = []
    for (day, ts), g in t.groupby(['day', 'timestamp'], sort=False):
        g = g.copy()
        if len(g) < 4:
            continue

        sig = g[signal_col]
        sig_z = (sig - sig.mean()) / (sig.std(ddof=0) if sig.std(ddof=0) > 0 else np.nan)
        sig_z = sig_z.replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-2.5, 2.5)

        # Mean-revert residual: short rich (+resid), long cheap (-resid).
        w_raw = -sig_z.to_numpy(dtype=float)

        delta = g['delta'].to_numpy(dtype=float)
        den = float(np.dot(delta, delta))
        if den <= 1e-12:
            continue

        lam = float(np.dot(w_raw, delta) / den)
        w_dn = w_raw - lam * delta
        if np.all(np.abs(w_dn) < 1e-10):
            continue

        # Normalize gross exposure then convert to contracts
        gross = np.sum(np.abs(w_dn))
        if gross <= 1e-12:
            continue

        w_dn = w_dn / gross
        target = contracts_scale * w_dn

        tmp = g[['day', 'timestamp', 'product', 'spread', 'ret_1']].copy()
        tmp['target_pos'] = target
        tmp['delta'] = delta
        rows.append(tmp)

    if not rows:
        return pd.DataFrame(), pd.DataFrame()

    pos = pd.concat(rows, ignore_index=True).sort_values(['day', 'product', 'timestamp'])
    pos['prev_target'] = pos.groupby(['day', 'product'])['target_pos'].shift(1).fillna(0.0)
    pos['turnover'] = (pos['target_pos'] - pos['prev_target']).abs()

    pos['gross_tick'] = pos['target_pos'] * pos['ret_1']
    pos['cost_tick'] = turnover_cost_weight * pos['turnover'] * (pos['spread'] / 2.0).abs().fillna(0.0)
    pos['net_tick'] = pos['gross_tick'] - pos['cost_tick']

    port = pos.groupby(['day', 'timestamp'], as_index=False).agg(
        gross_tick=('gross_tick', 'sum'),
        net_tick=('net_tick', 'sum'),
        gross_delta=('target_pos', 'sum'),
        turnover=('turnover', 'sum'),
    )
    return pos, port

pos_dn, port_dn = delta_neutral_smile_portfolio(df_alpha, signal_col='smile_resid')
if port_dn.empty:
    print('Delta-neutral smile portfolio did not produce enough valid rows.')
else:
    std_g = port_dn['gross_tick'].std(ddof=0)
    std_n = port_dn['net_tick'].std(ddof=0)

    print('Delta-neutral smile portfolio summary')
    print(f"rows={len(port_dn):,}")
    print(f"gross_pnl={port_dn['gross_tick'].sum():,.2f} | net_pnl={port_dn['net_tick'].sum():,.2f}")
    print(f"gross_sharpe={(port_dn['gross_tick'].mean()/std_g) if std_g>0 else np.nan:.4f}")
    print(f"net_sharpe={(port_dn['net_tick'].mean()/std_n) if std_n>0 else np.nan:.4f}")
    print(f"avg_turnover={port_dn['turnover'].mean():.4f}")

    print('\nTop products by net contribution:')
    contrib = pos_dn.groupby('product', as_index=False)['net_tick'].sum().sort_values('net_tick', ascending=False)
    display(contrib)
    display(port_dn.head(20))

Delta-neutral smile portfolio summary
rows=29,997
gross_pnl=70,605.56 | net_pnl=-658,140.73
gross_sharpe=0.2889
net_sharpe=-0.9633
avg_turnover=34.0983

Top products by net contribution:


,product,net_tick
8,VEV_6000,-17276.710384
9,VEV_6500,-18493.551037
5,VEV_5300,-18707.119573
1,VEV_4500,-24298.760492
0,VEV_4000,-37684.117766
7,VEV_5500,-57702.071788
4,VEV_5200,-59227.091034
6,VEV_5400,-82121.511535
3,VEV_5100,-129865.656451
2,VEV_5000,-212764.137693


,day,timestamp,gross_tick,net_tick,gross_delta,turnover
0,0,0.0,2.709357,-43.768302,6.522777,80.000000
1,0,100.0,-1.949443,-93.238171,-13.296254,84.267127
2,0,200.0,7.689139,-26.032552,5.934638,71.739750
3,0,300.0,-6.112663,-16.694575,6.630065,13.441414
4,0,400.0,24.379702,14.586028,5.934638,13.441414
5,0,500.0,5.230563,-15.618368,7.895193,31.461326
6,0,600.0,0.000000,-6.108009,7.493516,10.714543
7,0,700.0,-1.105959,-1.105959,7.493516,0.000000
8,0,800.0,7.442132,-2.903996,7.335350,17.571393
9,0,900.0,-11.511269,-21.023452,7.382134,11.531179


In [23]:
# Profitability survival grid: liquid + near-ATM + no-trade bands + passive vs crossing costs
if 'df_alpha' not in globals() or df_alpha.empty:
    raise ValueError('Run prior cells first so df_alpha is available.')

work = df_alpha.copy()
work = work.dropna(subset=['ret_1', 'spread', 'moneyness']).copy()

# Keep only liquid symbols and near-ATM region where spreads/slippage are usually better.
liq_stats = (
    work.groupby('product').agg(
        med_spread=('spread', 'median'),
        med_bidv=('bid_volume_1', 'median'),
        med_askv=('ask_volume_1', 'median'),
        rows=('ret_1', 'size'),
    ).reset_index()
 )
liq_stats['med_l1_vol'] = 0.5 * (liq_stats['med_bidv'].fillna(0) + liq_stats['med_askv'].fillna(0))

eligible_products = liq_stats[
    (liq_stats['rows'] > 15000)
    & (liq_stats['med_l1_vol'] >= 15)
].sort_values(['med_spread', 'med_l1_vol'], ascending=[True, False])['product'].tolist()

if not eligible_products:
    eligible_products = liq_stats.sort_values(['med_spread', 'rows']).head(4)['product'].tolist()

work = work[work['product'].isin(eligible_products)].copy()
work = work[work['moneyness'].abs() <= 0.08].copy()

if work.empty:
    raise ValueError('Filters are too strict; no data left. Relax moneyness/liquidity constraints.')

# Candidate options-focused signals + two blended variants.
signal_defs = {
    'iv_rv_mr': -work['iv_rv_spread'],
    'smile_resid_mr': -work['smile_resid'],
    'delta_hedge_resid_mr': -work['delta_hedged_resid'],
}

for k, v in signal_defs.items():
    work[k] = v

# Z-score each signal per (day, product).
for name in list(signal_defs.keys()):
    g = work[name].groupby([work['day'], work['product']])
    mu = g.transform('mean')
    sd = g.transform('std').replace(0, np.nan)
    work[name + '_z'] = ((work[name] - mu) / sd).clip(-3, 3).fillna(0.0)

work['blend_iv_smile_z'] = 0.6 * work['iv_rv_mr_z'] + 0.4 * work['smile_resid_mr_z']
work['blend_smile_delta_z'] = 0.6 * work['smile_resid_mr_z'] + 0.4 * work['delta_hedge_resid_mr_z']

signal_cols = [
    'iv_rv_mr_z',
    'smile_resid_mr_z',
    'delta_hedge_resid_mr_z',
    'blend_iv_smile_z',
    'blend_smile_delta_z',
]

def build_hysteresis_positions(sig: pd.Series, day: pd.Series, product: pd.Series, entry: float, exit_: float, cap: float) -> pd.Series:
    tmp = pd.DataFrame({'sig': sig, 'day': day, 'product': product}).copy()
    tmp = tmp.sort_index()

    def _run_group(g: pd.DataFrame) -> pd.Series:
        pos = 0.0
        out = []
        for s in g['sig'].to_numpy():
            if abs(s) >= entry:
                pos = float(np.clip(s, -cap, cap))
            elif abs(s) <= exit_:
                pos = 0.0
            out.append(pos)
        return pd.Series(out, index=g.index)

    return tmp.groupby(['day', 'product'], group_keys=False).apply(_run_group).sort_index()

def evaluate_combo(base: pd.DataFrame, sig_col: str, entry: float, exit_: float, spread_q: float, cap: float, passive_fill: float) -> dict:
    t = base[['day', 'timestamp', 'product', 'ret_1', 'spread', sig_col]].copy()
    t = t.dropna(subset=[sig_col, 'ret_1', 'spread']).copy()
    if t.empty:
        return {}

    # Regime filter: only trade when spread is in lower quantiles for that symbol/day.
    thr = t.groupby(['day', 'product'])['spread'].transform(lambda s: s.quantile(spread_q))
    active = t['spread'] <= thr

    t = t.sort_values(['day', 'product', 'timestamp']).copy()
    t['raw_pos'] = build_hysteresis_positions(t[sig_col], t['day'], t['product'], entry, exit_, cap)
    t['position'] = np.where(active, t['raw_pos'], 0.0)
    t['prev_pos'] = t.groupby(['day', 'product'])['position'].shift(1).fillna(0.0)
    t['turnover'] = (t['position'] - t['prev_pos']).abs()

    t['gross_tick'] = t['position'] * t['ret_1']
    t['half_spread'] = (t['spread'] / 2.0).abs()

    # Two cost lenses:
    # 1) Crossing model (worst-case aggressive)
    # 2) Passive model (effective cost reduced by fill probability)
    t['cost_cross'] = t['turnover'] * t['half_spread']
    t['cost_passive'] = (1.0 - passive_fill) * t['turnover'] * t['half_spread']

    t['net_cross'] = t['gross_tick'] - t['cost_cross']
    t['net_passive'] = t['gross_tick'] - t['cost_passive']

    std_g = t['gross_tick'].std(ddof=0)
    std_c = t['net_cross'].std(ddof=0)
    std_p = t['net_passive'].std(ddof=0)

    return {
        'signal': sig_col,
        'entry': entry,
        'exit': exit_,
        'spread_q': spread_q,
        'cap': cap,
        'passive_fill': passive_fill,
        'rows': int(len(t)),
        'gross_pnl': float(t['gross_tick'].sum()),
        'net_cross_pnl': float(t['net_cross'].sum()),
        'net_passive_pnl': float(t['net_passive'].sum()),
        'gross_sharpe': float(t['gross_tick'].mean() / std_g) if std_g > 0 else np.nan,
        'cross_sharpe': float(t['net_cross'].mean() / std_c) if std_c > 0 else np.nan,
        'passive_sharpe': float(t['net_passive'].mean() / std_p) if std_p > 0 else np.nan,
        'avg_turnover': float(t['turnover'].mean()),
        'active_rate': float((t['position'] != 0).mean()),
    }

entries = [1.0, 1.25, 1.5, 2.0]
exits = [0.25, 0.5, 0.75, 1.0]
spread_qs = [0.35, 0.45, 0.55]
caps = [1.0, 1.5]
passive_fills = [0.5, 0.7, 0.85]

grid_rows = []
for sig in signal_cols:
    for e in entries:
        for x in exits:
            if x >= e:
                continue
            for q in spread_qs:
                for c in caps:
                    for pf in passive_fills:
                        out = evaluate_combo(work, sig, e, x, q, c, pf)
                        if out:
                            grid_rows.append(out)

grid = pd.DataFrame(grid_rows)
if grid.empty:
    raise ValueError('Grid produced no rows.')

print('Eligible symbols (liquid + near-ATM universe):', eligible_products)
print(f'Rows after filters: {len(work):,}')

print('\nTop configs by passive net PnL')
top_passive = grid.sort_values(['net_passive_pnl', 'passive_sharpe'], ascending=False).head(20)
display(top_passive)

print('\nTop configs by crossing net PnL (harder benchmark)')
top_cross = grid.sort_values(['net_cross_pnl', 'cross_sharpe'], ascending=False).head(20)
display(top_cross)

best_passive = top_passive.iloc[0]
best_cross = top_cross.iloc[0]
print('\nBest passive config:')
print(best_passive.to_string())

print('\nBest crossing config:')
print(best_cross.to_string())

C:\Users\tolan\AppData\Local\Temp\ipykernel_28420\3590682615.py:76: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return tmp.groupby(['day', 'product'], group_keys=False).apply(_run_group).sort_index()
C:\Users\tolan\AppData\Local\Temp\ipykernel_28420\3590682615.py:76: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return tmp.groupby(['day', 'product'], group_keys=False).apply(_run_group).sort_index()
C:\Users\tolan\A

Eligible symbols (liquid + near-ATM universe): ['VEV_5400', 'VEV_5500', 'VEV_6000', 'VEV_6500', 'VEV_5300', 'VEV_5200', 'VEV_5100']
Rows after filters: 149,985

Top configs by passive net PnL


,signal,entry,exit,spread_q,cap,passive_fill,rows,gross_pnl,net_cross_pnl,net_passive_pnl,gross_sharpe,cross_sharpe,passive_sharpe,avg_turnover,active_rate
1064,blend_iv_smile_z,2.0,1.00,0.35,1.0,0.85,149985,40.50,-1247.0,-152.6250,0.003694,-0.060730,-0.013609,0.006667,0.015888
1070,blend_iv_smile_z,2.0,1.00,0.45,1.0,0.85,149985,40.50,-1247.0,-152.6250,0.003694,-0.060730,-0.013609,0.006667,0.015888
1076,blend_iv_smile_z,2.0,1.00,0.55,1.0,0.85,149985,40.50,-1247.0,-152.6250,0.003694,-0.060730,-0.013609,0.006667,0.015888
1046,blend_iv_smile_z,2.0,0.75,0.35,1.0,0.85,149985,65.00,-1409.0,-156.1000,0.005127,-0.062187,-0.012052,0.007561,0.021489
1052,blend_iv_smile_z,2.0,0.75,0.45,1.0,0.85,149985,65.00,-1409.0,-156.1000,0.005127,-0.062187,-0.012052,0.007561,0.021489
1058,blend_iv_smile_z,2.0,0.75,0.55,1.0,0.85,149985,65.00,-1409.0,-156.1000,0.005127,-0.062187,-0.012052,0.007561,0.021489
1028,blend_iv_smile_z,2.0,0.50,0.35,1.0,0.85,149985,61.00,-1648.0,-195.3500,0.004327,-0.066694,-0.013568,0.008761,0.027363
1034,blend_iv_smile_z,2.0,0.50,0.45,1.0,0.85,149985,61.00,-1648.0,-195.3500,0.004327,-0.066694,-0.013568,0.008761,0.027363
1040,blend_iv_smile_z,2.0,0.50,0.55,1.0,0.85,149985,61.00,-1648.0,-195.3500,0.004327,-0.066694,-0.013568,0.008761,0.027363
1067,blend_iv_smile_z,2.0,1.00,0.35,1.5,0.85,149985,60.75,-1870.5,-228.9375,0.003694,-0.060730,-0.013609,0.010001,0.015888



Top configs by crossing net PnL (harder benchmark)


,signal,entry,exit,spread_q,cap,passive_fill,rows,gross_pnl,net_cross_pnl,net_passive_pnl,gross_sharpe,cross_sharpe,passive_sharpe,avg_turnover,active_rate
1062,blend_iv_smile_z,2.0,1.00,0.35,1.0,0.50,149985,40.5,-1247.0,-603.250,0.003694,-0.060730,-0.043396,0.006667,0.015888
1063,blend_iv_smile_z,2.0,1.00,0.35,1.0,0.70,149985,40.5,-1247.0,-345.750,0.003694,-0.060730,-0.028676,0.006667,0.015888
1064,blend_iv_smile_z,2.0,1.00,0.35,1.0,0.85,149985,40.5,-1247.0,-152.625,0.003694,-0.060730,-0.013609,0.006667,0.015888
1068,blend_iv_smile_z,2.0,1.00,0.45,1.0,0.50,149985,40.5,-1247.0,-603.250,0.003694,-0.060730,-0.043396,0.006667,0.015888
1069,blend_iv_smile_z,2.0,1.00,0.45,1.0,0.70,149985,40.5,-1247.0,-345.750,0.003694,-0.060730,-0.028676,0.006667,0.015888
1070,blend_iv_smile_z,2.0,1.00,0.45,1.0,0.85,149985,40.5,-1247.0,-152.625,0.003694,-0.060730,-0.013609,0.006667,0.015888
1074,blend_iv_smile_z,2.0,1.00,0.55,1.0,0.50,149985,40.5,-1247.0,-603.250,0.003694,-0.060730,-0.043396,0.006667,0.015888
1075,blend_iv_smile_z,2.0,1.00,0.55,1.0,0.70,149985,40.5,-1247.0,-345.750,0.003694,-0.060730,-0.028676,0.006667,0.015888
1076,blend_iv_smile_z,2.0,1.00,0.55,1.0,0.85,149985,40.5,-1247.0,-152.625,0.003694,-0.060730,-0.013609,0.006667,0.015888
1044,blend_iv_smile_z,2.0,0.75,0.35,1.0,0.50,149985,65.0,-1409.0,-672.000,0.005127,-0.062187,-0.042748,0.007561,0.021489



Best passive config:
signal             blend_iv_smile_z
entry                           2.0
exit                            1.0
spread_q                       0.35
cap                             1.0
passive_fill                   0.85
rows                         149985
gross_pnl                      40.5
net_cross_pnl               -1247.0
net_passive_pnl            -152.625
gross_sharpe               0.003694
cross_sharpe               -0.06073
passive_sharpe            -0.013609
avg_turnover               0.006667
active_rate                0.015888

Best crossing config:
signal             blend_iv_smile_z
entry                           2.0
exit                            1.0
spread_q                       0.35
cap                             1.0
passive_fill                    0.5
rows                         149985
gross_pnl                      40.5
net_cross_pnl               -1247.0
net_passive_pnl             -603.25
gross_sharpe               0.003694
cross_sharpe       

In [24]:
# Other options signals: surface dynamics, cross-strike stress, and convexity errors
if 'df_alpha' not in globals() or df_alpha.empty:
    raise ValueError('Run earlier cells first to build df_alpha.')

other = df_alpha.copy()
other = other.sort_values(['day', 'timestamp', 'product']).copy()

# Keep tradable core universe around ATM plus liquid wings.
liq = other.groupby('product').agg(
    rows=('ret_1', 'size'),
    med_spread=('spread', 'median'),
    med_bid=('bid_volume_1', 'median'),
    med_ask=('ask_volume_1', 'median'),
 ).reset_index()
liq['med_l1'] = 0.5 * (liq['med_bid'].fillna(0) + liq['med_ask'].fillna(0))

keep_products = liq[(liq['rows'] > 12000) & (liq['med_l1'] >= 12)].sort_values(['med_spread', 'med_l1'], ascending=[True, False])['product'].tolist()
if not keep_products:
    keep_products = liq.sort_values(['med_spread', 'rows']).head(6)['product'].tolist()

other = other[other['product'].isin(keep_products)].copy()
other = other[other['moneyness'].abs() <= 0.12].copy()

if other.empty:
    raise ValueError('No rows after filtering; relax filters.')

gdp = other.groupby(['day', 'product'])
gdt = other.groupby(['day', 'timestamp'])

# 1) IV momentum and acceleration (surface dynamics per strike)
other['d_iv_1'] = gdp['iv'].diff(1)
other['d_iv_3'] = gdp['iv'].diff(3)
other['d2_iv'] = other['d_iv_1'] - gdp['d_iv_1'].shift(1)

# 2) Skew and curvature dynamics by timestamp
other['skew_proxy'] = np.where(other['moneyness'].abs() > 1e-8, other['smile_resid'] / other['moneyness'], np.nan)
other['curv_proxy'] = other['smile_resid'] / (other['moneyness'].abs() + 1e-4)

other['skew_mean_ts'] = gdt['skew_proxy'].transform('mean')
other['curv_mean_ts'] = gdt['curv_proxy'].transform('mean')
other['skew_disp_ts'] = gdt['skew_proxy'].transform(lambda s: s.std(ddof=0))
other['curv_disp_ts'] = gdt['curv_proxy'].transform(lambda s: s.std(ddof=0))

# 3) Cross-strike pressure: rank-normalized residual at each timestamp
other['xrank_resid'] = gdt['smile_resid'].transform(lambda s: s.rank(pct=True) - 0.5)

# 4) Convexity error: realized move vs implied move through gamma
other['convexity_err'] = other['gamma'] * ((other['spot_ret_1'] ** 2) - (other['implied_move'] ** 2))

# 5) Spot-vol lead/lag interaction
other['spot_iv_interact'] = other['spot_ret_1'] * gdp['iv'].diff(1)

# Build additional signal candidates
sig_map = {
    'iv_mom_revert': -other['d_iv_1'],
    'iv_accel_revert': -other['d2_iv'],
    'skew_shift_mom': other['skew_mean_ts'],
    'curv_shift_mom': other['curv_mean_ts'],
    'skew_disp_revert': -other['skew_disp_ts'],
    'curv_disp_revert': -other['curv_disp_ts'],
    'xrank_resid_revert': -other['xrank_resid'],
    'convexity_err_mom': other['convexity_err'],
    'spot_iv_interact_mr': -other['spot_iv_interact'],
}

for name, s in sig_map.items():
    other[name] = s

# Z-score by (day, product) for consistency with earlier evaluator
for name in sig_map.keys():
    gg = other[name].groupby([other['day'], other['product']])
    mu = gg.transform('mean')
    sd = gg.transform('std').replace(0, np.nan)
    other[name + '_z'] = ((other[name] - mu) / sd).clip(-3, 3).fillna(0.0)

other_signals = [k + '_z' for k in sig_map.keys()]

rows = []
for s in other_signals:
    r1 = evaluate_candidate(other, s, horizon_col='ret_1', entry=1.1, exit_=0.6, pos_cap=1.2, spread_cost_weight=0.7)
    r1['horizon'] = 'ret_1'
    rows.append(r1)

    r2 = evaluate_candidate(other, s, horizon_col='ret_2', entry=1.1, exit_=0.6, pos_cap=1.2, spread_cost_weight=0.7)
    r2['horizon'] = 'ret_2'
    rows.append(r2)

other_summary = pd.DataFrame(rows).sort_values(['net_pnl', 'net_sharpe'], ascending=False).reset_index(drop=True)

print('Other-signal universe products:', keep_products)
print(f'Rows after filters: {len(other):,}')
print('\nTop other-signal configs')
display(other_summary.head(25))

if not other_summary.empty:
    best_other = other_summary.iloc[0]
    print('\nBest new signal candidate:')
    print(best_other.to_string())

Other-signal universe products: ['VEV_5400', 'VEV_5500', 'VEV_6000', 'VEV_6500', 'VEV_5300', 'VEV_5200', 'VEV_5100']
Rows after filters: 149,985

Top other-signal configs


,alpha,rows,gross_pnl,net_pnl,gross_sharpe,net_sharpe,turnover,active_rate,horizon
0,skew_disp_revert_z,149970,217.886706,-11269.578134,0.003205,-0.138051,0.094861,0.273655,ret_2
1,skew_disp_revert_z,149985,-32.678975,-11460.383856,-0.000631,-0.165728,0.094344,0.272827,ret_1
2,curv_disp_revert_z,149970,508.617330,-14360.091762,0.007158,-0.164034,0.124324,0.301560,ret_2
3,curv_disp_revert_z,149985,301.179190,-14542.881408,0.005552,-0.193933,0.124081,0.301397,ret_1
4,curv_shift_mom_z,149985,413.635177,-27723.617441,0.007823,-0.323983,0.233947,0.276794,ret_1
5,curv_shift_mom_z,149970,159.126534,-28015.631367,0.002327,-0.290667,0.234280,0.277055,ret_2
6,convexity_err_mom_z,149970,-407.373946,-28528.303297,-0.005538,-0.283648,0.226396,0.210242,ret_2
7,convexity_err_mom_z,149985,-487.106434,-28613.498384,-0.007902,-0.309854,0.226413,0.210228,ret_1
8,skew_shift_mom_z,149985,-933.534237,-37226.028736,-0.015111,-0.374736,0.294184,0.397240,ret_1
9,skew_shift_mom_z,149970,-1265.707692,-37568.287881,-0.015564,-0.334192,0.294325,0.397680,ret_2



Best new signal candidate:
alpha           skew_disp_revert_z
rows                        149970
gross_pnl               217.886706
net_pnl              -11269.578134
gross_sharpe              0.003205
net_sharpe               -0.138051
turnover                  0.094861
active_rate               0.273655
horizon                      ret_2


In [25]:
# Rare-trade survival search: extreme thresholds + tight spread regime
if 'other' not in globals() or other.empty:
    raise ValueError('Run the previous Other options signals cell first.')

rare = other.copy()
rare = rare.sort_values(['day', 'product', 'timestamp']).copy()

candidate_cols = [
    'skew_disp_revert_z',
    'curv_disp_revert_z',
    'spot_iv_interact_mr_z',
    'convexity_err_mom_z',
]
candidate_cols = [c for c in candidate_cols if c in rare.columns]
if not candidate_cols:
    raise ValueError('No candidate signal columns found.')

def eval_rare(df: pd.DataFrame, sig_col: str, entry: float, exit_: float, spread_q: float, cap: float, cost_w: float):
    t = df[['day', 'timestamp', 'product', 'ret_1', 'spread', sig_col]].dropna().copy()
    if t.empty:
        return None

    # Only trade during very tight spread moments per day/product.
    spread_thr = t.groupby(['day', 'product'])['spread'].transform(lambda s: s.quantile(spread_q))
    t['allowed'] = t['spread'] <= spread_thr

    t['sig'] = t[sig_col]
    t['raw_pos'] = 0.0

    def _hyst(g: pd.DataFrame) -> pd.Series:
        pos = 0.0
        out = []
        for s in g['sig'].to_numpy():
            if abs(s) >= entry:
                pos = float(np.clip(s, -cap, cap))
            elif abs(s) <= exit_:
                pos = 0.0
            out.append(pos)
        return pd.Series(out, index=g.index)

    t['raw_pos'] = t.groupby(['day', 'product'], group_keys=False).apply(_hyst)
    t['position'] = np.where(t['allowed'], t['raw_pos'], 0.0)
    t['prev'] = t.groupby(['day', 'product'])['position'].shift(1).fillna(0.0)
    t['turnover'] = (t['position'] - t['prev']).abs()

    t['gross'] = t['position'] * t['ret_1']
    t['net'] = t['gross'] - cost_w * t['turnover'] * (t['spread'] / 2.0).abs().fillna(0.0)

    std = t['net'].std(ddof=0)
    return {
        'signal': sig_col,
        'entry': entry,
        'exit': exit_,
        'spread_q': spread_q,
        'cap': cap,
        'cost_w': cost_w,
        'rows': int(len(t)),
        'net_pnl': float(t['net'].sum()),
        'net_sharpe': float(t['net'].mean() / std) if std > 0 else np.nan,
        'turnover': float(t['turnover'].mean()),
        'active_rate': float((t['position'] != 0).mean()),
    }

rows = []
for s in candidate_cols:
    for e in [1.75, 2.0, 2.25, 2.5, 3.0]:
        for x in [0.25, 0.5, 0.75, 1.0, 1.25]:
            if x >= e:
                continue
            for q in [0.15, 0.2, 0.25, 0.3]:
                for cap in [0.75, 1.0]:
                    for cw in [0.7, 0.85, 1.0]:
                        out = eval_rare(rare, s, e, x, q, cap, cw)
                        if out is not None:
                            rows.append(out)

rare_grid = pd.DataFrame(rows)
rare_grid = rare_grid.sort_values(['net_pnl', 'net_sharpe'], ascending=False).reset_index(drop=True)

print('Rare-trade search top results')
display(rare_grid.head(25))

print('Configs with very low activity (< 6%)')
low_act = rare_grid[rare_grid['active_rate'] < 0.06].sort_values(['net_pnl', 'net_sharpe'], ascending=False)
display(low_act.head(20))

if not rare_grid.empty:
    print('Best rare-trade config:')
    print(rare_grid.iloc[0].to_string())

C:\Users\tolan\AppData\Local\Temp\ipykernel_28420\2454706291.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  t['raw_pos'] = t.groupby(['day', 'product'], group_keys=False).apply(_hyst)
C:\Users\tolan\AppData\Local\Temp\ipykernel_28420\2454706291.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  t['raw_pos'] = t.groupby(['day', 'product'], group_keys=False).apply(_hyst)
C:\Users\tolan\AppData\Local\Temp\ipyker

Rare-trade search top results


C:\Users\tolan\AppData\Local\Temp\ipykernel_28420\2454706291.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  t['raw_pos'] = t.groupby(['day', 'product'], group_keys=False).apply(_hyst)


,signal,entry,exit,spread_q,cap,cost_w,rows,net_pnl,net_sharpe,turnover,active_rate
0,curv_disp_revert_z,3.0,1.25,0.15,0.75,0.70,149985,-250.01250,-0.035920,0.00226,0.007021
1,skew_disp_revert_z,3.0,1.00,0.15,0.75,0.70,149985,-252.30000,-0.036845,0.00207,0.006021
2,skew_disp_revert_z,3.0,1.25,0.15,0.75,0.70,149985,-258.82500,-0.037708,0.00212,0.005927
3,curv_disp_revert_z,3.0,1.00,0.15,0.75,0.70,149985,-261.71250,-0.035603,0.00227,0.007587
4,skew_disp_revert_z,3.0,1.00,0.20,0.75,0.70,149985,-273.82500,-0.038612,0.00219,0.006121
5,skew_disp_revert_z,3.0,1.00,0.25,0.75,0.70,149985,-273.82500,-0.038612,0.00219,0.006121
6,skew_disp_revert_z,3.0,1.00,0.30,0.75,0.70,149985,-273.82500,-0.038612,0.00219,0.006121
7,curv_disp_revert_z,3.0,1.25,0.20,0.75,0.70,149985,-274.50000,-0.037861,0.00239,0.007141
8,curv_disp_revert_z,3.0,1.25,0.25,0.75,0.70,149985,-274.50000,-0.037861,0.00239,0.007141
9,curv_disp_revert_z,3.0,1.25,0.30,0.75,0.70,149985,-274.50000,-0.037861,0.00239,0.007141


Configs with very low activity (< 6%)


,signal,entry,exit,spread_q,cap,cost_w,rows,net_pnl,net_sharpe,turnover,active_rate
0,curv_disp_revert_z,3.0,1.25,0.15,0.75,0.70,149985,-250.0125,-0.035920,0.00226,0.007021
1,skew_disp_revert_z,3.0,1.00,0.15,0.75,0.70,149985,-252.3000,-0.036845,0.00207,0.006021
2,skew_disp_revert_z,3.0,1.25,0.15,0.75,0.70,149985,-258.8250,-0.037708,0.00212,0.005927
3,curv_disp_revert_z,3.0,1.00,0.15,0.75,0.70,149985,-261.7125,-0.035603,0.00227,0.007587
4,skew_disp_revert_z,3.0,1.00,0.20,0.75,0.70,149985,-273.8250,-0.038612,0.00219,0.006121
5,skew_disp_revert_z,3.0,1.00,0.25,0.75,0.70,149985,-273.8250,-0.038612,0.00219,0.006121
6,skew_disp_revert_z,3.0,1.00,0.30,0.75,0.70,149985,-273.8250,-0.038612,0.00219,0.006121
7,curv_disp_revert_z,3.0,1.25,0.20,0.75,0.70,149985,-274.5000,-0.037861,0.00239,0.007141
8,curv_disp_revert_z,3.0,1.25,0.25,0.75,0.70,149985,-274.5000,-0.037861,0.00239,0.007141
9,curv_disp_revert_z,3.0,1.25,0.30,0.75,0.70,149985,-274.5000,-0.037861,0.00239,0.007141


Best rare-trade config:
signal         curv_disp_revert_z
entry                         3.0
exit                         1.25
spread_q                     0.15
cap                          0.75
cost_w                        0.7
rows                       149985
net_pnl                 -250.0125
net_sharpe               -0.03592
turnover                  0.00226
active_rate              0.007021
